In [1]:
!pip install gymnasium
!pip install stable_baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [36]:
import pandas as pd

rye_microgrid_data = pd.read_csv('train.csv')
rye_microgrid_data

,time,pv_production,wind_production,consumption,spot_market_price,precip_1h:mm,precip_type:idx,prob_precip_1h:p,clear_sky_rad:W,clear_sky_energy_1h:J,...,t_50m:C,relative_humidity_50m:p,dew_point_50m:C,wind_speed_50m:ms,wind_dir_50m:d,t_100m:C,relative_humidity_100m:p,dew_point_100m:C,wind_speed_100m:ms,wind_dir_100m:d
0,2020-01-01 13:00:00,0.0,40.59,26.514689,0.28969,0.0,0.0,1.0,10.0,64826.0,...,8.4,60.7,1.3,8.4,246.3,8.3,60.3,1.0,10.4,247.3
1,2020-01-01 14:00:00,0.0,67.86,28.326960,0.29561,0.0,0.0,1.0,0.0,8961.1,...,8.4,61.6,1.5,8.0,252.3,8.4,60.7,1.2,10.0,252.1
2,2020-01-01 15:00:00,0.0,116.68,23.682207,0.30044,0.0,0.0,1.0,0.0,0.0,...,8.5,60.3,1.3,9.6,254.1,8.4,59.6,1.0,11.7,253.8
3,2020-01-01 16:00:00,0.0,120.22,25.354782,0.29975,0.0,0.0,1.0,0.0,0.0,...,8.4,63.9,2.0,12.1,254.6,8.3,63.4,1.7,14.3,254.2
4,2020-01-01 17:00:00,0.0,109.86,23.861942,0.29650,0.0,0.0,1.0,0.0,0.0,...,7.2,78.9,3.8,11.7,249.5,7.1,77.9,3.5,13.9,249.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9510,2021-01-31 19:00:00,0.0,21.98,44.422658,0.60243,0.0,0.0,1.0,0.0,0.0,...,-3.2,93.1,-4.2,4.8,231.2,-3.1,91.6,-4.3,6.0,238.3
9511,2021-01-31 20:00:00,0.0,9.60,45.167707,0.53335,0.0,0.0,1.0,0.0,0.0,...,-3.5,94.1,-4.3,4.9,224.9,-3.3,92.1,-4.4,6.2,231.8
9512,2021-01-31 21:00:00,0.0,22.61,32.476198,0.51195,0.0,0.0,1.0,0.0,0.0,...,-3.6,93.0,-4.6,4.7,224.5,-3.3,90.6,-4.6,6.0,231.7
9513,2021-01-31 22:00:00,0.0,21.70,28.561791,0.47122,0.0,0.0,1.0,0.0,0.0,...,-3.7,92.1,-4.8,4.5,222.8,-3.4,89.5,-4.9,5.8,231.1


Simpler Version: Similar to Nick and Luthira's versions

In [42]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces

class RyeMicrogridEnvSimple(gym.Env):
  """
    Custom Microgrid Environment for Rye Microgrid
  """
  def __init__(self, data, num_days):
    super(RyeMicrogridEnvSimple, self).__init__()
    # no of hours to simulate
    self.num_hours = 24 * num_days
    self.curr_hour = 0

    # store all necessary data for Rye Microgrid
    self.data = data

    # setup battery
    self.battery_max = 100
    self.battery_power = 0.5 * self.battery_max

    # State consists of PV, Wind, Battery, Load and Price
    self.observation_space = spaces.Box(low=0, high=1, shape=(5,), dtype=np.float32)

    # Action Space:
    self.action_space = spaces.Discrete(4)

    # charge/discharge rate
    # self.battery_use_rate = 10

    self.state = self._get_observation()

  def _get_observation(self):
    # Returns the state variables
    curr_data = self.data.loc[self.curr_hour, ['pv_production', 'wind_production', 'consumption', 'spot_market_price']]

    return np.array([curr_data['pv_production'], curr_data['wind_production'], self.battery_power, curr_data['consumption'], curr_data['spot_market_price']], dtype=np.float32)

  def step(self, action):
    # Unpack state
    pv, wind, battery, load, price = self.state

    # Set reward and done states
    reward = 0
    done = False
    consumption = load - pv - wind

    if action == 0: # Charge ESS
      if consumption < 0:
        if abs(consumption) > self.battery_max - self.battery_power:
          self.battery_power = 100
        else:
          self.battery_power -= consumption
      else:
        reward -= 100 * consumption
    elif action == 1: # Discharge ESS
      if consumption > 0:
        consumption -= self.battery_power
        if consumption > 0:
          self.battery_power = 0
          reward -= 100 * consumption
        else:
          self.battery_power = -consumption
    elif action == 2: # Buy from grid
      if consumption > 0:
        reward -= consumption * price
    elif action == 3: # Sell to grid
      if consumption < 0:
        reward -= consumption * price
      else:
        reward -= 100 * consumption

    # Update state
    self.curr_hour += 1
    if self.curr_hour >= self.num_hours:
        done = True

    self.state = self._get_observation()
    return self.state, reward, done, False, {}

  def reset(self, seed=None, options=None):
    # Reset important parameters
    super().reset(seed=seed)
    self.curr_hour = 0
    self.battery_power = 0.5 * self.battery_max
    self.state = self._get_observation()
    return self.state, {}



In [45]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env

# Create and wrap environment
env = RyeMicrogridEnvSimple(rye_microgrid_data, num_days=50)
vec_env = make_vec_env(lambda: env, n_envs=1)

# Train the PPO agent
model = DQN("MlpPolicy", vec_env, verbose=1)
model.learn(total_timesteps=100000)

# Save the model
model.save("dqn_microgrid-simple")

Using cpu device
-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 1.2e+03   |
|    ep_rew_mean      | -6.58e+05 |
|    exploration_rate | 0.544     |
| time/               |           |
|    episodes         | 4         |
|    fps              | 836       |
|    time_elapsed     | 5         |
|    total_timesteps  | 4800      |
| train/              |           |
|    learning_rate    | 0.0001    |
|    loss             | 635       |
|    n_updates        | 1174      |
-----------------------------------
-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 1.2e+03   |
|    ep_rew_mean      | -4.51e+05 |
|    exploration_rate | 0.088     |
| time/               |           |
|    episodes         | 8         |
|    fps              | 760       |
|    time_elapsed     | 12        |
|    total_timesteps  | 9600      |
| train/              |           |
|    learning_rate    | 0.0001    |
|    loss  

In [43]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env

# Create and wrap environment
env = RyeMicrogridEnvSimple(rye_microgrid_data, num_days=7)
vec_env = make_vec_env(lambda: env, n_envs=1)

# Train the PPO agent
model = DQN("MlpPolicy", vec_env, verbose=1)
model.learn(total_timesteps=10000)

# Save the model
model.save("dqn_microgrid-simple")

Using cpu device
-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 168       |
|    ep_rew_mean      | -1.45e+05 |
|    exploration_rate | 0.362     |
| time/               |           |
|    episodes         | 4         |
|    fps              | 940       |
|    time_elapsed     | 0         |
|    total_timesteps  | 672       |
| train/              |           |
|    learning_rate    | 0.0001    |
|    loss             | 715       |
|    n_updates        | 142       |
-----------------------------------
-----------------------------------
| rollout/            |           |
|    ep_len_mean      | 168       |
|    ep_rew_mean      | -1.08e+05 |
|    exploration_rate | 0.05      |
| time/               |           |
|    episodes         | 8         |
|    fps              | 868       |
|    time_elapsed     | 1         |
|    total_timesteps  | 1344      |
| train/              |           |
|    learning_rate    | 0.0001    |
|    loss  

I also tried some PPO stuff with more complex action spaces but it didnt give as good results. I think its because of the way I am setting the weights as my reward is based on price (energy * price/kw) based on if renewable is used (+ve) or grid is used (-ve). I also set -1000 as I didnt want any consumption load unattended or if its charging/selling with battery at same time. I think I made this model too complex and messed up something in my reward logic so will try to simplify it.

In [37]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces

class RyeMicrogridEnv(gym.Env):
  """
    Custom Microgrid Environment for Rye Microgrid
  """
  def __init__(self, data, num_days):
    super(RyeMicrogridEnv, self).__init__()
    # no of hours to simulate
    self.num_hours = 24 * num_days
    self.curr_hour = 0

    # store all necessary data for Rye Microgrid
    self.data = data

    # setup battery
    self.battery_max = 100
    self.battery_power = 0.5 * self.battery_max

    # State consists of PV, Wind, Battery, Load and Price
    self.observation_space = spaces.Box(low=0, high=1, shape=(5,), dtype=np.float32)

    # Action Space: Solar (Charge ESS, Satisfy Load, Sell to Grid), Wind (Charge ESS, Satisfy Load, Sell to Grid), Grid (Connect, Disconnect), Battery (Charge, Discharge and Satisfy Load, Sell to Grid)
    self.action_space = spaces.MultiDiscrete([3, 3, 2, 3])

    self.state = self._get_observation()

  def _get_observation(self):
    # Returns the state variables
    curr_data = self.data.loc[self.curr_hour, ['pv_production', 'wind_production', 'consumption', 'spot_market_price']]

    return np.array([curr_data['pv_production'], curr_data['wind_production'], self.battery_power, curr_data['consumption'], curr_data['spot_market_price']], dtype=np.float32)

  def step(self, action):
    # Unpack state
    pv, wind, battery, load, price = self.state
    solar_act, wind_act, grid_act, battery_act = action

    # Set reward and done states
    reward = 0
    done = False
    consumption = load

    # Action Handling
    # For Solar
    if solar_act == 0:
      if self.battery_power == self.battery_max:
        reward -= pv * price
      else:
        if pv > self.battery_max - self.battery_power:
          self.battery_power = self.battery_max
          reward += price * (- pv + self.battery_max - self.battery_power)
        else:
          self.battery_power += pv
          # reward += 10 * pv
    elif solar_act == 1:
      if pv > consumption:
        reward += consumption * price
        consumption = 0
      else:
        reward += price * (consumption - pv)
        consumption -= pv
    elif solar_act == 2:
      reward += price * pv

    # For Wind
    if wind_act == 0:
      if self.battery_power == self.battery_max:
        reward -= wind * price
      else:
        if wind > self.battery_max - self.battery_power:
          self.battery_power = self.battery_max
          reward += price * (- wind + self.battery_max - self.battery_power)
        else:
          self.battery_power += wind
          # reward += 10 * wind
    elif wind_act == 1:
      if wind > consumption:
        reward += consumption * price
        consumption = 0
      else:
        reward += price * (consumption - wind)
        consumption -= wind
    elif wind_act == 2:
      reward += price * wind

    # For Battery
    if battery_act == 0:
      pass
    elif battery_act == 1:
      if solar_act == 0 or wind_act == 0:
        if self.battery_power > consumption:
          self.battery_power -= consumption
          reward += price * consumption
          consumption = 0
        elif self.battery_power < consumption:
          reward += (consumption-self.battery_power) * 10
          consumption -= self.battery_power
          self.battery_power = 0
      else:
        reward -= 1000
    elif battery_act == 2:
      if solar_act != 0 and wind_act != 0:
        if self.battery_power > 0.5 * self.battery_max:
          reward += price * (self.battery_power-0.5*self.battery_max)
          self.battery_power -= (self.battery_power-0.5*self.battery_max)
      else:
        reward -= 1000

    # For Grid
    if grid_act == 0:
      if consumption > 0:
        reward -= 1000 * consumption
    elif grid_act == 1:
      if consumption > 0:
        reward -= price * consumption

    # Update state
    self.curr_hour += 1
    if self.curr_hour >= self.num_hours:
        done = True

    self.state = self._get_observation()
    return self.state, reward, done, False, {}

  def reset(self, seed=None, options=None):
    # Reset important parameters
    super().reset(seed=seed)
    self.curr_hour = 0
    self.battery_power = 0.5 * self.battery_max
    self.state = self._get_observation()
    return self.state, {}



In [38]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Create and wrap environment
env = RyeMicrogridEnv(rye_microgrid_data, num_days=100)
vec_env = make_vec_env(lambda: env, n_envs=1)

# Train the PPO agent
model = PPO("MlpPolicy", vec_env, verbose=1)
model.learn(total_timesteps=10000)


# Save the model
model.save("dqn_microgrid")

Using cpu device
-----------------------------
| time/              |      |
|    fps             | 481  |
|    iterations      | 1    |
|    time_elapsed    | 4    |
|    total_timesteps | 2048 |
-----------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 2.4e+03       |
|    ep_rew_mean          | -1.86e+07     |
| time/                   |               |
|    fps                  | 415           |
|    iterations           | 2             |
|    time_elapsed         | 9             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 2.1509826e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -3.99         |
|    explained_variance   | 5.66e-06      |
|    learning_rate        | 0.0003        |
|    loss                 | 9.57e+09      |
|    n_updates           